# Offline teacher distillation (Colab)

Build distilled SFT JSONL for RAFT-LM Unsloth training.

**Output:** `train.jsonl`, `val.jsonl`, `test.jsonl`, `manifest.json` → download and place under `data/distilled/<corpus_name>/`.

**Default teacher:** `Qwen/Qwen3-4B-Instruct-2507`

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import json
import random
from pathlib import Path

TEACHER_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
CORPUS_NAME = "risk_sft_v1"
OUTPUT_DIR = Path(f"/content/distilled/{CORPUS_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS = [
    {"record_id": "p001", "prompt": "Assess liquidity risk for a portfolio with 40% illiquid assets.", "risk_domain": "liquidity"},
    {"record_id": "p002", "prompt": "Summarize market risk under calm conditions.", "risk_domain": "market"},
    {"record_id": "p003", "prompt": "Evaluate operational risk after a control failure.", "risk_domain": "operational"},
]
print(f"{len(PROMPTS)} benchmark prompts loaded")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

In [ ]:
def generate_completion(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

rows = []
for item in PROMPTS:
    completion = generate_completion(item["prompt"])
    rows.append({
        "record_id": item["record_id"],
        "prompt": item["prompt"],
        "completion": completion,
        "risk_domain": item["risk_domain"],
        "engine_version": "colab-distill-v1",
    })
rows[:1]

In [ ]:
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

random.seed(42)
shuffled = rows.copy()
random.shuffle(shuffled)
n = len(shuffled)
train = shuffled[: max(1, int(0.7 * n))]
val = shuffled[max(1, int(0.7 * n)) : max(2, int(0.85 * n))] or train[:1]
test = shuffled[max(2, int(0.85 * n)) :] or val[:1]

write_jsonl(OUTPUT_DIR / "train.jsonl", train)
write_jsonl(OUTPUT_DIR / "val.jsonl", val)
write_jsonl(OUTPUT_DIR / "test.jsonl", test)

manifest = {
    "corpus_id": CORPUS_NAME,
    "teacher_model": TEACHER_MODEL,
    "source": "colab_distillation",
    "counts": {"train": len(train), "val": len(val), "test": len(test)},
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote splits to {OUTPUT_DIR}")

## Export

1. Zip `OUTPUT_DIR` and download from Colab.
2. Unpack into `data/distilled/<corpus_name>/` in the RAFT-LM repo.
3. Train: `python scripts/train.py --config configs/training/unsloth_lora_example.yaml`